# Week 12 Lab — Big Data Tools
**SQL · APIs · PySpark**  
AUCA · Introduction to Big Data Analytics · November 2026

**Student name:** *(replace with your name)*  
**Files needed:** `week12_transactions.csv` · `week12_api_sample.json`

In [23]:
# ── Colab only: install PySpark (takes 2-3 min) ──────────────────────────────
# Uncomment the line below when running on Google Colab:
# !pip install pyspark

# ── Standard imports used throughout ─────────────────────────────────────────
import sqlite3
import json
import pandas as pd

print('Libraries imported ✓')

Libraries imported ✓


---
## Part 1 · SQL with Python  *(12 marks)*

We load `week12_transactions.csv` into an **SQLite** in-memory database, then answer each question with `pd.read_sql(query, conn)`.

In [24]:
# ── Setup: load CSV → SQLite ──────────────────────────────────────────────────
df = pd.read_csv('week12_transactions.csv')
conn = sqlite3.connect('momo.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)

print(f'Loaded {len(df):,} rows into momo.db  ✓')
pd.read_sql('SELECT * FROM transactions LIMIT 5', conn)

Loaded 2,000 rows into momo.db  ✓


,txn_id,date,hour,district,txn_type,amount_rwf,flagged
0,T00001,2026-11-01,8,Gasabo,send,1000,0
1,T00002,2026-11-09,12,Nyarugenge,send,25000,0
2,T00003,2026-11-19,8,Muhanga,send,51000,0
3,T00004,2026-11-01,17,Huye,send,79000,0
4,T00005,2026-11-15,12,Huye,send,109000,0


### Exercise 1.1 — SELECT / WHERE / ORDER BY  *(4 marks)*

In [25]:
# ── 1.1 (a) Top-5 'send' transactions above 100,000 RWF ──────────────────────
q1a = """
SELECT txn_id, district, amount_rwf
FROM   transactions
WHERE  txn_type = 'send'
  AND  amount_rwf > 100000
ORDER BY amount_rwf DESC
LIMIT 5;
"""

result_1a = pd.read_sql(q1a, conn)
print('Top 5 send transactions above 100,000 RWF:')
result_1a

Top 5 send transactions above 100,000 RWF:


,txn_id,district,amount_rwf
0,T01708,Kicukiro,140000
1,T00364,Huye,139000
2,T00895,Muhanga,139000
3,T00899,Nyarugenge,139000
4,T01314,Gasabo,139000


In [26]:
# ── 1.1 (b) Count flagged transactions (flagged = 1) ─────────────────────────
q1b = """
SELECT COUNT(*) AS flagged_count
FROM   transactions
WHERE  flagged = 1;
"""

result_1b = pd.read_sql(q1b, conn)
print('Flagged transaction count:')
print(result_1b)
print()
print('✓ Checkpoint: should be 11 →', result_1b.iloc[0,0])

Flagged transaction count:
   flagged_count
0             11

✓ Checkpoint: should be 11 → 11


### Exercise 1.2 — GROUP BY  *(5 marks)*

In [27]:
# ── 1.2 (a) Per txn_type: COUNT, SUM, AVG — ordered by total descending ──────
q2a = """
SELECT   txn_type,
         COUNT(*)               AS n_transactions,
         SUM(amount_rwf)        AS total_rwf,
         ROUND(AVG(amount_rwf)) AS avg_rwf
FROM     transactions
GROUP BY txn_type
ORDER BY total_rwf DESC;
"""

result_2a = pd.read_sql(q2a, conn)
print('Amount per transaction type (highest total first):')
display(result_2a)

# Format for readability
result_2a['total_rwf_M'] = (result_2a['total_rwf'] / 1_000_000).round(1)
top_type = result_2a.iloc[0]
print(f"\n→ '{top_type['txn_type']}' moves the most money: {top_type['total_rwf_M']:.1f}M RWF")
print('✓ Checkpoint: send ~62.0M RWF')

Amount per transaction type (highest total first):


,txn_type,n_transactions,total_rwf,avg_rwf
0,send,868,62049000,71485.0
1,paybill,330,29970000,90818.0
2,receive,489,27024000,55264.0
3,airtime,313,833000,2661.0



→ 'send' moves the most money: 62.0M RWF
✓ Checkpoint: send ~62.0M RWF


In [28]:
# ── 1.2 (b) Total per district — which district leads? ───────────────────────
q2b = """
SELECT   district,
         COUNT(*)        AS n_transactions,
         SUM(amount_rwf) AS total_rwf
FROM     transactions
GROUP BY district
ORDER BY total_rwf DESC;
"""

result_2b = pd.read_sql(q2b, conn)
result_2b['total_rwf_M'] = (result_2b['total_rwf'] / 1_000_000).round(2)
print('Total amount per district (highest first):')
display(result_2b)

leader = result_2b.iloc[0]
print(f"\n→ Leading district: {leader['district']} with {leader['total_rwf_M']:.2f}M RWF")
print('✓ Checkpoint: Muhanga leads with ~21.1M RWF')

Total amount per district (highest first):


,district,n_transactions,total_rwf,total_rwf_M
0,Muhanga,247,21076000,21.08
1,Gasabo,255,20738000,20.74
2,Huye,251,18736000,18.74
3,Nyarugenge,252,18091000,18.09
4,Kicukiro,249,11338000,11.34
5,Musanze,253,10490500,10.49
6,Rubavu,246,10110500,10.11
7,Nyagatare,247,9296000,9.30



→ Leading district: Muhanga with 21.08M RWF
✓ Checkpoint: Muhanga leads with ~21.1M RWF


### Exercise 1.3 — The busiest hour  *(3 marks)*

In [29]:
# ── 1.3 (a) Transactions per hour in SQL — busiest first ─────────────────────
q3a = """
SELECT   hour,
         COUNT(*) AS n_transactions
FROM     transactions
GROUP BY hour
ORDER BY n_transactions DESC;
"""

result_3a = pd.read_sql(q3a, conn)
print('Transactions per hour (SQL), top 5:')
print(result_3a.head())

busiest_sql = result_3a.iloc[0]
print(f"\n→ SQL busiest hour: hour {int(busiest_sql['hour'])} with {int(busiest_sql['n_transactions'])} transactions")

Transactions per hour (SQL), top 5:
   hour  n_transactions
0    20             185
1    17             183
2    18             176
3    12             170
4    11             169

→ SQL busiest hour: hour 20 with 185 transactions


In [30]:
# ── 1.3 (b) Same question in Pandas — both must agree ────────────────────────
hourly_pandas = (
    df.groupby('hour')['txn_id']
      .count()
      .rename('n_transactions')
      .sort_values(ascending=False)
      .reset_index()
)

print('Transactions per hour (Pandas), top 5:')
print(hourly_pandas.head())

busiest_pd = hourly_pandas.iloc[0]
print(f"\n→ Pandas busiest hour: hour {int(busiest_pd['hour'])} with {int(busiest_pd['n_transactions'])} transactions")

# Confirm agreement
assert int(busiest_sql['hour']) == int(busiest_pd['hour']), 'Mismatch!'
assert int(busiest_sql['n_transactions']) == int(busiest_pd['n_transactions']), 'Mismatch!'
print()
print('✓ SQL and Pandas agree!')
print('✓ Checkpoint: hour 20 with 185 transactions')

Transactions per hour (Pandas), top 5:
   hour  n_transactions
0    20             185
1    17             183
2    18             176
3    12             170
4    11             169

→ Pandas busiest hour: hour 20 with 185 transactions

✓ SQL and Pandas agree!
✓ Checkpoint: hour 20 with 185 transactions


---
## Part 2 · APIs & JSON  *(10 marks)*

JSON responses from APIs are just Python dicts and lists — the same structures from Week 1.

### Exercise 2.1 — JSON by hand  *(3 marks)*

In [31]:
# ── 2.1 Parse a JSON string manually ─────────────────────────────────────────
json_string = '{"city": "Kigali", "readings": [21.5, 24.0, 26.2, 23.8, 22.1]}'

data = json.loads(json_string)   # text → Python dict

# (a) City
city = data['city']
print(f'(a) City: {city}')

# (b) Number of readings
n_readings = len(data['readings'])
print(f'(b) Number of readings: {n_readings}')

# (c) Average (2 decimal places)
avg_temp = sum(data['readings']) / len(data['readings'])
print(f'(c) Average temperature: {avg_temp:.2f} °C')

(a) City: Kigali
(b) Number of readings: 5
(c) Average temperature: 23.52 °C


### Exercise 2.2 — Call a real API  *(4 marks)*

In [32]:
# ── 2.2 Fetch Kigali's 24-hour forecast from Open-Meteo ──────────────────────
#    Falls back to week12_api_sample.json if offline.

try:
    import requests
    url = (
        'https://api.open-meteo.com/v1/forecast'
        '?latitude=-1.9441&longitude=30.0619'
        '&hourly=temperature_2m&forecast_days=1'
    )
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    api_data = r.json()
    print('Live API call succeeded ✓  (status', r.status_code, ')')
except Exception as e:
    print(f'Live API unavailable ({e}) — using offline sample.')
    with open('week12_api_sample.json') as f:
        api_data = json.load(f)

# Extract time and temperature lists
times = api_data['hourly']['time'][:24]
temps = api_data['hourly']['temperature_2m'][:24]

# (a) How many hours
print(f'(a) Number of hourly readings: {len(temps)}')

# (b) Max and min temperature
max_temp = max(temps)
min_temp = min(temps)
print(f'(b) Max temperature: {max_temp} °C')
print(f'    Min temperature: {min_temp} °C')

# (c) Time of the hottest hour
hottest_time = times[temps.index(max(temps))]
print(f'(c) Hottest hour: {hottest_time}')

Live API call succeeded ✓  (status 200 )
(a) Number of hourly readings: 24
(b) Max temperature: 26.2 °C
    Min temperature: 14.9 °C
(c) Hottest hour: 2026-11-25T12:00


### Exercise 2.3 — API data meets your toolbox  *(3 marks)*

In [33]:
# ── 2.3 Build a DataFrame from the API data ───────────────────────────────────
weather_df = pd.DataFrame({'time': times, 'temp_c': temps})
print('Weather DataFrame (first 5 rows):')
print(weather_df.head())
print()

Weather DataFrame (first 5 rows):
               time  temp_c
0  2026-11-25T00:00    16.0
1  2026-11-25T01:00    15.4
2  2026-11-25T02:00    15.1
3  2026-11-25T03:00    14.9
4  2026-11-25T04:00    14.9



In [34]:
# ── 2.3 (a) describe() the temperature ───────────────────────────────────────
print('Temperature statistics:')
print(weather_df['temp_c'].describe().round(2))

Temperature statistics:
count    24.00
mean     19.94
std       3.74
min      14.90
25%      17.08
50%      19.35
75%      22.85
max      26.20
Name: temp_c, dtype: float64


In [35]:
# ── 2.3 (b) Count hours >= 25 °C with a mask ─────────────────────────────────
hot_mask = weather_df['temp_c'] >= 25
hot_hours = weather_df[hot_mask]

print(f'Hours at or above 25 °C: {len(hot_hours)}')
print()
print('Those hours:')
print(hot_hours.to_string(index=False))
print()
print('✓ Checkpoint: 9 hours >= 25 °C (sample file) →', len(hot_hours))

Hours at or above 25 °C: 3

Those hours:
            time  temp_c
2026-11-25T10:00    25.5
2026-11-25T11:00    25.9
2026-11-25T12:00    26.2

✓ Checkpoint: 9 hours >= 25 °C (sample file) → 3


---
## Part 3 · PySpark  *(10 marks)*

We run Spark in **local mode** — your machine simulates a cluster. The code is identical to what runs on a real thousand-node cluster.

In [36]:
# ── Setup: start SparkSession ─────────────────────────────────────────────────
# Startup warnings about native libraries are normal — ignore them.
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('Week12_MoMo')
    .master('local[*]')
    .config('spark.driver.memory', '2g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')   # silence INFO/WARN spam
print('SparkSession ready ✓')
print('Spark version:', spark.version)

SparkSession ready ✓
Spark version: 4.3.0


### Exercise 3.1 — Load & inspect  *(3 marks)*

In [37]:
# ── 3.1 Load CSV into Spark DataFrame ────────────────────────────────────────
sdf = spark.read.csv(
    'week12_transactions.csv',
    header=True,
    inferSchema=True
)

# Row count
print(f'Row count: {sdf.count():,}')
print()

# Schema (like df.info() in Pandas)
print('Schema:')
sdf.printSchema()

Row count: 2,000

Schema:
root
 |-- txn_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)



In [38]:
# First 5 rows
print('First 5 rows:')
sdf.show(5)

First 5 rows:
+------+----------+----+----------+--------+----------+-------+
|txn_id|      date|hour|  district|txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00001|2026-11-01|   8|    Gasabo|    send|      1000|      0|
|T00002|2026-11-09|  12|Nyarugenge|    send|     25000|      0|
|T00003|2026-11-19|   8|   Muhanga|    send|     51000|      0|
|T00004|2026-11-01|  17|      Huye|    send|     79000|      0|
|T00005|2026-11-15|  12|      Huye|    send|    109000|      0|
+------+----------+----+----------+--------+----------+-------+
only showing top 5 rows


### Exercise 3.2 — Filter  *(3 marks)*

In [39]:
# ── 3.2 (a) Count transactions above 100,000 RWF ────────────────────────────
big_txns = sdf.filter(sdf.amount_rwf > 100_000)
count_big = big_txns.count()
print(f'Transactions above 100,000 RWF: {count_big:,}')
print('✓ Checkpoint: should be 451 →', count_big)

Transactions above 100,000 RWF: 451
✓ Checkpoint: should be 451 → 451


In [40]:
# ── 3.2 (b) Show 5 flagged transactions ──────────────────────────────────────
print('5 flagged transactions (flagged = 1):')
sdf.filter(sdf.flagged == 1).show(5)

5 flagged transactions (flagged = 1):
+------+----------+----+--------+--------+----------+-------+
|txn_id|      date|hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00279|2026-11-29|  19| Muhanga|    send|    127000|      1|
|T00326|2026-11-07|  22|    Huye|    send|    127000|      1|
|T00536|2026-11-27|  17|  Gasabo|    send|    127000|      1|
|T00655|2026-11-30|  17|  Rubavu|    send|    134000|      1|
|T00697|2026-11-01|   9| Muhanga|    send|    127000|      1|
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


### Exercise 3.3 — groupBy  *(4 marks)*

In [41]:
# ── 3.3 (a) Total amount per txn_type ────────────────────────────────────────
print('Total amount_rwf per transaction type (Spark):')
sdf.groupBy('txn_type').sum('amount_rwf').orderBy('sum(amount_rwf)', ascending=False).show()

Total amount_rwf per transaction type (Spark):
+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
|    send|       62049000|
| paybill|       29970000|
| receive|       27024000|
| airtime|         833000|
+--------+---------------+



In [42]:
# ── 3.3 (b) Per-district totals → convert to Pandas and sort ─────────────────
district_spark = (
    sdf.groupBy('district')
       .sum('amount_rwf')
       .withColumnRenamed('sum(amount_rwf)', 'total_rwf')
)

district_pd = district_spark.toPandas().sort_values('total_rwf', ascending=False).reset_index(drop=True)
district_pd['total_rwf_M'] = (district_pd['total_rwf'] / 1_000_000).round(2)
print('Per-district totals (Spark → Pandas):')
print(district_pd.to_string(index=False))

/projects/Big_Data/week 12/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/projects/Big_Data/week 12/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Per-district totals (Spark → Pandas):
  district  total_rwf  total_rwf_M
   Muhanga   21076000        21.08
    Gasabo   20738000        20.74
      Huye   18736000        18.74
Nyarugenge   18091000        18.09
  Kicukiro   11338000        11.34
   Musanze   10490500        10.49
    Rubavu   10110500        10.11
 Nyagatare    9296000         9.30


In [43]:
# ── 3.3 (c) Do Spark answers match SQL answers? ───────────────────────────────
# Compare top type
spark_top_type_total = (
    sdf.groupBy('txn_type').sum('amount_rwf')
       .orderBy('sum(amount_rwf)', ascending=False)
       .toPandas()
)

print('SQL top type total (send):', result_2a.iloc[0]['total_rwf'])
print('Spark top type total (send):', spark_top_type_total.iloc[0]['sum(amount_rwf)'])

print()
print('SQL top district total (Muhanga):', result_2b.iloc[0]['total_rwf'])
print('Spark top district total (Muhanga):', district_pd.iloc[0]['total_rwf'])

# Comment: YES — Spark answers match SQL answers exactly.
# This is expected: both compute the same aggregations on the same data.
# The only difference is WHERE the computation runs (SQLite engine vs Spark engine).
print()
print('✓ Spark and SQL answers match!')

/projects/Big_Data/week 12/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/projects/Big_Data/week 12/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


SQL top type total (send): 62049000
Spark top type total (send): 62049000

SQL top district total (Muhanga): 21076000
Spark top district total (Muhanga): 21076000

✓ Spark and SQL answers match!


### Bonus (+3) — Spark speaks SQL too

In [44]:
# ── Bonus: register the view and run Part 1 SQL unchanged on Spark ────────────
sdf.createOrReplaceTempView('transactions')

# Same GROUP BY query as Part 1.2 (a) — zero changes
spark_sql_result = spark.sql("""
    SELECT   txn_type,
             COUNT(*)               AS n_transactions,
             SUM(amount_rwf)        AS total_rwf,
             ROUND(AVG(amount_rwf)) AS avg_rwf
    FROM     transactions
    GROUP BY txn_type
    ORDER BY total_rwf DESC
""")

print('Spark SQL result — same query as Part 1.2 (a):')
spark_sql_result.show()

# Comment: Why is it powerful that the same SQL runs on SQLite and on a
# thousand-machine cluster?


Spark SQL result — same query as Part 1.2 (a):
+--------+--------------+---------+-------+
|txn_type|n_transactions|total_rwf|avg_rwf|
+--------+--------------+---------+-------+
|    send|           868| 62049000|71485.0|
| paybill|           330| 29970000|90818.0|
| receive|           489| 27024000|55264.0|
| airtime|           313|   833000| 2661.0|
+--------+--------------+---------+-------+



---
## Part 4 · Choose Your Weapon  *(3 marks — written)*

For each scenario, name the first tool you would reach for and justify in one sentence.

| # | Scenario | Tool | Justification |
|---|----------|------|---------------|
| 1 | Ministry of Health gives read-access to their PostgreSQL database and asks for monthly totals per district. | **SQL** | The data already lives in a relational database, so SQL is the native language of that system — it queries the data in place without moving it. |
| 2 | Your project group has a 40,000-row CSV needing cleaning, three charts, and a regression model. | **Pandas** | 40,000 rows fits comfortably in memory, and Pandas integrates directly with Matplotlib/Plotly for charts and Scikit-learn for modelling in one workflow. |
| 3 | A telecom wants nationwide call-record analysis — 40 TB already on their cluster. | **Spark** | 40 TB will not fit in any single machine's RAM; Spark is designed to distribute that computation across the cluster where the data already lives. |
| 4 | Your dashboard must always show today's weather without anyone uploading a file. | **API** | A weather API (like Open-Meteo) delivers the current forecast on demand, so the dashboard stays live without any manual data updates. |

---
## Part 5 · Final-Project Proposal

**Group D — Road Accident Risk Analysis in Rwanda**

Our group will investigate what patterns are associated with road accidents in Rwanda and whether available data can predict accident severity or risk. We will use a road-safety dataset of at least 500 real accident records — preferably sourced from the Rwanda National Police or a comparable open road-safety repository — containing attributes such as location, time of day, road and weather conditions, vehicle type, and injury outcome. Road accidents are a leading cause of preventable death and injury in Rwanda and across sub-Saharan Africa, so understanding when and where severe accidents cluster can directly inform traffic enforcement, infrastructure investment, and public-awareness campaigns. We plan to train a **classification model** (starting with logistic regression and comparing with a random forest) to predict accident severity class, evaluated using a confusion matrix, recall, precision, and F1-score, because the target is categorical and class imbalance between minor and fatal incidents makes recall on the severe class the most critical metric. Findings will be communicated through an interactive Plotly dashboard highlighting risk patterns and practical road-safety recommendations.